# Clase 6 — Integración liviana de PLN + IA

En las clases anteriores ya trabajamos:

- carga de un modelo de Hugging Face;
- configuración de temperatura, tokens y otros parámetros;
- prompts de sistema;
- historial de conversación.

En esta clase vamos a integrar el PLN del primer cuatrimestre, pero de forma **más liviana**.

La idea importante es:

> La IA ya entiende bastante bien los errores normales de escritura. Por eso no necesitamos hacer un corrector ortográfico completo.

Lo que sí vamos a conservar de PLN:

- normalización mínima para analizar el texto;
- detección de saludo;
- detección de tono;
- detección de intenciones;
- uso del análisis como pista para el modelo;
- registro de lo que el chatbot detectó.

Cambio importante de esta versión:

> El tono detectado no queda solo como dato decorativo. Se convierte en una instrucción concreta dentro del prompt para que la IA adapte la forma de responder.

La pregunta original del usuario se manda al modelo. El texto normalizado se usa solo para analizar, comparar y explicar qué detectó el chatbot.

Antes de ejecutar: activar **GPU T4** en Colab.  
Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4.

## 1. Instalación liviana

No vamos a instalar Pandas ni fijar versiones innecesarias.  
Esta celda solo instala librerías si no están disponibles en el entorno.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = [
    ("transformers", "transformers>=4.51,<5"),
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("gradio", "gradio")
]

faltan = []

for modulo, paquete in paquetes:
    if importlib.util.find_spec(modulo) is None:
        faltan.append(paquete)

if faltan:
    print("Faltan paquetes. Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + faltan)
else:
    print("Todos los paquetes necesarios ya están instalados.")


In [ ]:
import re
import time
import unicodedata

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("PyTorch:", torch.__version__)
print("Gradio:", gr.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Activá GPU T4 antes de continuar.")


## 2. Así configuré mi modelo

Cada grupo debería empezar a definir su propia configuración.  
Estos valores están pensados para clase: respuestas más rápidas y no tan largas.


In [ ]:
CONFIG_MODELO = {
    "nombre_chatbot": "AulaIA",
    "modelo": "Qwen/Qwen2.5-1.5B-Instruct",
    "temperature": 0.3,
    "top_p": 0.85,
    "top_k": 20,
    "max_new_tokens": 180,
    "max_mensajes_historial": 6
}

CONFIG_MODELO


### ¿Qué significa esta configuración?

- `modelo`: modelo generativo que vamos a usar.
- `temperature`: controla la creatividad. Para un chatbot institucional conviene un valor bajo.
- `top_p` y `top_k`: limitan las opciones que el modelo considera al generar.
- `max_new_tokens`: limita la extensión de la respuesta. Menos tokens suele ser más rápido.
- `max_mensajes_historial`: cantidad de mensajes anteriores que se envían como contexto.


## 3. Cargar el modelo

Usamos un modelo más liviano que el final para que las pruebas de clase sean más ágiles.


In [ ]:
MODELO_ID = CONFIG_MODELO["modelo"]

config_cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Cargando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

print("Cargando modelo...")
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    quantization_config=config_cuantizacion,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

modelo.eval()
print("Modelo cargado correctamente.")


## 4. Función rápida para generar respuestas

Esta función recibe los mensajes, los manda al modelo y mide cuánto tarda.


In [ ]:
def generar_respuesta(mensajes, config=None):
    if config is None:
        config = CONFIG_MODELO

    texto_chat = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto_chat,
        return_tensors="pt"
    ).to(modelo.device)

    inicio = time.time()

    temperature = float(config.get("temperature", 0.3))
    do_sample = temperature > 0

    argumentos_generacion = {
        "max_new_tokens": int(config.get("max_new_tokens", 180)),
        "do_sample": do_sample,
        "repetition_penalty": 1.08,
        "pad_token_id": tokenizer.eos_token_id
    }

    if do_sample:
        argumentos_generacion["temperature"] = temperature
        argumentos_generacion["top_p"] = float(config.get("top_p", 0.85))
        argumentos_generacion["top_k"] = int(config.get("top_k", 20))

    with torch.inference_mode():
        salida = modelo.generate(
            **entradas,
            **argumentos_generacion
        )

    tiempo = round(time.time() - inicio, 2)

    tokens_nuevos = salida[0][entradas["input_ids"].shape[-1]:]
    respuesta = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)

    return respuesta.strip(), tiempo


## 5. PLN mínimo que vamos a conservar

No vamos a usar un diccionario grande de correcciones.  
La normalización se usa para analizar el texto, no para reemplazar la pregunta original.


In [ ]:
def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto)
    return "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )


def reducir_repeticiones(texto):
    # Reduce exageraciones como "holaaaa" a "holaa".
    # No intenta corregir todas las palabras.
    return re.sub(r"(.)\1{2,}", r"\1\1", texto)


def normalizar_para_analisis(texto):
    texto = str(texto).strip().lower()
    texto = quitar_tildes(texto)
    texto = reducir_repeticiones(texto)
    texto = re.sub(r"[^a-z0-9áéíóúñü\s¿?¡!.,]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


In [ ]:
ejemplos = [
    "Holaaaa, dnd queda y q requisitos necesito???",
    "Buenas tardes, quisiera saber cómo inscribirme.",
    "HACE RATO PREGUNTO EL HORARIO!!!"
]

for ejemplo in ejemplos:
    print("Original:   ", ejemplo)
    print("Para análisis:", normalizar_para_analisis(ejemplo))
    print("-" * 60)


## 6. Detección de saludo, tono e intenciones

Estas funciones no reemplazan la respuesta de la IA.  
Generan información adicional para orientar el comportamiento del chatbot.

En esta versión el **tono detectado sí se usa**:

- si el usuario escribe formal, la respuesta será más institucional;
- si escribe informal, será más cercana y simple;
- si está molesto, responderá con calma y empatía;
- si agradece, responderá de forma breve y cordial.

El objetivo es que los alumnos vean una integración real entre PLN e IA.

In [ ]:
PATRONES_INTENCIONES = {
    "inscripcion": [
        r"\binscrip", r"\banotarme\b", r"\bpreinscripcion\b", r"\bingreso\b"
    ],
    "requisitos": [
        r"\brequisit", r"\bdocument", r"\bpapeles\b", r"\bdni\b", r"\btitulo\b"
    ],
    "carreras": [
        r"\bcarreras?\b", r"\boferta academica\b", r"\bque puedo estudiar\b"
    ],
    "horarios": [
        r"\bhorarios?\b", r"\ba que hora\b", r"\bhasta que hora\b", r"\bcuando atienden\b"
    ],
    "ubicacion": [
        r"\bubicacion\b", r"\bdireccion\b", r"\bdonde queda\b", r"\bdnd\b", r"\bcomo llego\b"
    ],
    "costos": [
        r"\bprecio\b", r"\bcuota\b", r"\barancel", r"\bcosto\b", r"\bcuanto sale\b", r"\bes gratis\b"
    ],
    "agradecimiento": [
        r"\bgracias\b", r"\bmuchas gracias\b", r"\bgenial\b", r"\bexcelente\b"
    ]
}


def detectar_intenciones(texto_original):
    texto = normalizar_para_analisis(texto_original)
    intenciones = []

    for intencion, patrones in PATRONES_INTENCIONES.items():
        if any(re.search(patron, texto) for patron in patrones):
            intenciones.append(intencion)

    return intenciones


def detectar_saludo(texto_original):
    texto = normalizar_para_analisis(texto_original)

    saludos = [
        "buen dia", "buenos dias", "buenas tardes",
        "buenas noches", "hola", "holi", "buenas"
    ]

    for saludo in saludos:
        if saludo in texto:
            return saludo

    return None


def detectar_tono(texto_original):
    texto_original = str(texto_original)
    texto = normalizar_para_analisis(texto_original)

    molestias = [
        "hace rato", "nadie responde", "urgente", "no funciona",
        "una verguenza", "molesto", "enojado", "pesimo"
    ]

    agradecimientos = [
        "gracias", "muchas gracias", "te agradezco", "genial", "excelente", "muy amable"
    ]

    formalidad = [
        "buen dia", "buenos dias", "buenas tardes", "buenas noches",
        "quisiera", "podria", "por favor", "disculpe", "necesitaria"
    ]

    informalidad = [
        "hola", "holi", "buenas", "che", "porfa", "me decis", "dale"
    ]

    # La molestia tiene prioridad porque cambia mucho la forma de responder.
    if (
        any(frase in texto for frase in molestias)
        or texto_original.count("!") >= 2
        or (texto_original.isupper() and len(texto_original) > 8)
    ):
        return "molesto"

    if any(frase in texto for frase in agradecimientos):
        return "agradecido"

    if any(frase in texto for frase in formalidad):
        return "formal"

    if any(frase in texto for frase in informalidad):
        return "informal"

    return "neutral"


def generar_instruccion_tono(tono):
    """Convierte el tono detectado por PLN en una instrucción concreta para el modelo."""

    instrucciones = {
        "formal": (
            "Respondé con tono formal, claro e institucional. "
            "Usá frases completas y evitá expresiones demasiado informales."
        ),
        "informal": (
            "Respondé con tono cercano, simple y natural. "
            "Podés usar un estilo más conversacional, pero manteniendo claridad."
        ),
        "molesto": (
            "Respondé con calma, empatía y claridad. "
            "Reconocé la preocupación del usuario, no discutas y ofrecé pasos concretos."
        ),
        "agradecido": (
            "Respondé de forma breve, amable y cordial. "
            "No agregues información innecesaria si el usuario solo está agradeciendo."
        ),
        "neutral": (
            "Respondé con tono claro, educativo y amable. "
            "Mantené una explicación simple y ordenada."
        )
    }

    return instrucciones.get(tono, instrucciones["neutral"])


def analizar_pln(pregunta):
    tono = detectar_tono(pregunta)

    return {
        "pregunta_original": pregunta,
        "texto_para_analisis": normalizar_para_analisis(pregunta),
        "saludo": detectar_saludo(pregunta) or "ninguno",
        "tono": tono,
        "instruccion_tono": generar_instruccion_tono(tono),
        "intenciones": detectar_intenciones(pregunta) or ["no detectadas"]
    }


In [ ]:
for ejemplo in ejemplos:
    print(ejemplo)
    print(analizar_pln(ejemplo))
    print("-" * 60)


## 7. Responder usando IA + análisis de PLN

El análisis de PLN se agrega al prompt como información estructurada.  
La IA responde usando la pregunta original, no una versión corregida.

En esta parte se ve la integración real:

```text
pregunta original del usuario
↓
PLN detecta tono, saludo e intención
↓
el tono se transforma en instrucción de estilo
↓
la IA responde usando esa instrucción
```

Por ejemplo, si el tono detectado es `molesto`, el prompt le pide al modelo responder con calma y empatía.

In [ ]:
PROMPT_SISTEMA = """
Sos un asistente educativo de Procesamiento del Lenguaje Natural.
Respondé siempre en español claro y con un estilo adecuado para estudiantes de nivel terciario.

Reglas:
1. Usá la pregunta original del usuario como mensaje principal.
2. Usá el análisis de PLN solo como pista para ajustar tono, saludo e intención.
3. Aplicá la instrucción de tono indicada en cada consulta.
4. No digas que corregiste el texto del usuario.
5. No inventes datos institucionales exactos como fechas, precios, teléfonos, requisitos o reglamentos.
6. Si la pregunta es general, podés responder con conocimiento general.
7. Si falta información específica, pedí una aclaración.
8. Respondé breve, claro y útil.
""".strip()


def construir_mensaje_usuario(pregunta, analisis):
    intenciones = ", ".join(analisis["intenciones"])

    return f"""
PREGUNTA ORIGINAL DEL USUARIO:
{pregunta}

ANÁLISIS DE PLN:
- texto para análisis: {analisis['texto_para_analisis']}
- saludo detectado: {analisis['saludo']}
- tono detectado: {analisis['tono']}
- instrucción de tono: {analisis['instruccion_tono']}
- intenciones detectadas: {intenciones}

INSTRUCCIÓN:
Respondé a la pregunta original.
Aplicá la instrucción de tono detectada.
No menciones etiquetas internas como "tono", "intención" o "PLN", salvo que te pregunten específicamente por el análisis.
""".strip()


def recortar_historial(historial, max_mensajes):
    if historial is None:
        return []
    return historial[-int(max_mensajes):]


def conversar_con_pln(pregunta, historial=None, config=None):
    if config is None:
        config = CONFIG_MODELO

    if historial is None:
        historial = []

    analisis = analizar_pln(pregunta)
    mensaje_usuario = construir_mensaje_usuario(pregunta, analisis)

    historial_recortado = recortar_historial(
        historial,
        config.get("max_mensajes_historial", 6)
    )

    mensajes = [{"role": "system", "content": PROMPT_SISTEMA}]
    mensajes.extend(historial_recortado)
    mensajes.append({"role": "user", "content": mensaje_usuario})

    respuesta, tiempo = generar_respuesta(mensajes, config)

    historial.append({"role": "user", "content": pregunta})
    historial.append({"role": "assistant", "content": respuesta})

    return respuesta, historial, analisis, tiempo


## 8. Primera prueba sin interfaz

Esta prueba sirve para verificar que el modelo responde y que el análisis de PLN se genera correctamente.


In [ ]:
historial_prueba = []

pregunta = "Holaaaa, dnd queda y q requisitos necesito???"
respuesta, historial_prueba, analisis, tiempo = conversar_con_pln(pregunta, historial_prueba)

print("PREGUNTA")
print(pregunta)
print("\nANÁLISIS DE PLN")
print(analisis)
print("\nRESPUESTA")
print(respuesta)
print("\nTiempo:", tiempo, "segundos")


## 9. Interfaz básica para pruebas

Usamos una interfaz simple para que las pruebas no queden solo en consola.  
Esta versión evita el componente `Chatbot` para no depender del formato interno de Gradio.

La interfaz muestra:

- pregunta enviada;
- respuesta del modelo;
- análisis de PLN;
- tiempo de generación;
- configuración editable.


In [ ]:
def mostrar_cargando(pregunta):
    pregunta = "" if pregunta is None else str(pregunta).strip()

    if pregunta == "":
        return "", "Escribí una pregunta para probar el chatbot.", {}

    pregunta_visible = f"**Pregunta enviada:** {pregunta}"
    respuesta_visible = "⏳ Generando respuesta..."
    analisis_visible = {}

    return pregunta_visible, respuesta_visible, analisis_visible


def responder_interfaz(pregunta, historial, temperature, max_new_tokens, max_mensajes):
    pregunta = "" if pregunta is None else str(pregunta).strip()

    if historial is None:
        historial = []

    if pregunta == "":
        return "", "Escribí una pregunta para probar el chatbot.", {}, historial

    config_interfaz = CONFIG_MODELO.copy()
    config_interfaz["temperature"] = float(temperature)
    config_interfaz["max_new_tokens"] = int(max_new_tokens)
    config_interfaz["max_mensajes_historial"] = int(max_mensajes)

    respuesta, historial, analisis, tiempo = conversar_con_pln(
        pregunta,
        historial,
        config_interfaz
    )

    pregunta_visible = f"**Pregunta enviada:** {pregunta}"
    respuesta_visible = f"{respuesta}\n\n---\n_Tiempo de generación: {tiempo} segundos_"

    return pregunta_visible, respuesta_visible, analisis, historial


def limpiar_interfaz():
    return "", "", {}, []


with gr.Blocks() as demo:
    gr.Markdown("# Chatbot PLN + IA")
    gr.Markdown(
        "Probá cómo el chatbot usa la pregunta original y, además, analiza tono, saludo e intenciones. "
        "El tono detectado se transforma en una instrucción para la IA."
    )

    historial_estado = gr.State([])

    pregunta = gr.Textbox(
        label="Pregunta del usuario",
        placeholder="Ejemplo: Holaaaa, dnd queda y q requisitos necesito???",
        lines=2
    )

    with gr.Row():
        boton_enviar = gr.Button("Enviar", variant="primary")
        boton_limpiar = gr.Button("Limpiar")

    with gr.Accordion("Configuración del modelo", open=False):
        temperature = gr.Slider(
            minimum=0.0,
            maximum=1.0,
            value=CONFIG_MODELO["temperature"],
            step=0.1,
            label="Creatividad / temperature"
        )

        max_new_tokens = gr.Slider(
            minimum=50,
            maximum=300,
            value=CONFIG_MODELO["max_new_tokens"],
            step=10,
            label="Extensión máxima"
        )

        max_mensajes = gr.Slider(
            minimum=0,
            maximum=12,
            value=CONFIG_MODELO["max_mensajes_historial"],
            step=2,
            label="Mensajes de historial"
        )

    pregunta_mostrada = gr.Markdown(label="Pregunta enviada")
    respuesta_mostrada = gr.Markdown(label="Respuesta")
    analisis_mostrado = gr.JSON(label="Análisis de PLN usado en el prompt")

    gr.Markdown("### Preguntas sugeridas")
    gr.Markdown(
        """
        **Probá el mismo tema con tonos distintos:**
        - Buenas tardes, quisiera saber qué requisitos necesito para inscribirme.
        - hola, me decis q requisitos necesito?
        - HACE RATO PREGUNTO LOS REQUISITOS Y NADIE RESPONDE!!!
        - Muchas gracias, eso era todo.

        **Probá historial:**
        - Estoy armando un chatbot para una biblioteca.
        - Quiero que responda sobre préstamos y horarios.
        - ¿Qué tono debería usar para ese proyecto?
        - ¿Sobre qué proyecto te pregunté?
        """
    )

    pregunta.submit(
        fn=mostrar_cargando,
        inputs=[pregunta],
        outputs=[pregunta_mostrada, respuesta_mostrada, analisis_mostrado]
    ).then(
        fn=responder_interfaz,
        inputs=[pregunta, historial_estado, temperature, max_new_tokens, max_mensajes],
        outputs=[pregunta_mostrada, respuesta_mostrada, analisis_mostrado, historial_estado]
    )

    boton_enviar.click(
        fn=mostrar_cargando,
        inputs=[pregunta],
        outputs=[pregunta_mostrada, respuesta_mostrada, analisis_mostrado]
    ).then(
        fn=responder_interfaz,
        inputs=[pregunta, historial_estado, temperature, max_new_tokens, max_mensajes],
        outputs=[pregunta_mostrada, respuesta_mostrada, analisis_mostrado, historial_estado]
    )

    boton_limpiar.click(
        fn=limpiar_interfaz,
        inputs=[],
        outputs=[pregunta_mostrada, respuesta_mostrada, analisis_mostrado, historial_estado]
    )

demo.launch(debug=True)


## 10. Actividad de la clase

Cada grupo debe adaptar esta idea a su propio chatbot.

### Parte A — Configuración del modelo

Definan y anoten:

| Elemento | Valor elegido | Justificación |
|---|---|---|
| Nombre del chatbot | | |
| Modelo usado | | |
| Temperature | | |
| Máximo de tokens | | |
| Mensajes de historial | | |

### Parte B — Intenciones del proyecto

Definan al menos 5 intenciones propias de su chatbot.

Ejemplo para biblioteca:

- horarios;
- préstamos;
- devolución;
- búsqueda de libros;
- contacto.

### Parte C — Tono y respuesta

Prueben mensajes con distintos tonos y observen si la respuesta cambia:

1. mensaje formal;
2. mensaje informal;
3. mensaje con error o abreviación;
4. mensaje molesto;
5. mensaje de agradecimiento;
6. mensaje fuera de tema.

Para cada prueba observen:

- qué tono detectó;
- qué instrucción de tono generó;
- qué intención detectó;
- si la respuesta se adaptó bien;
- cuánto tardó en responder.

### Parte D — Historial

Prueben una conversación encadenada:

1. Estoy armando un chatbot para una biblioteca.
2. Quiero que responda sobre préstamos y horarios.
3. ¿Qué tono debería usar para ese proyecto?
4. ¿Sobre qué proyecto te pregunté?

Luego cambien el valor de mensajes de historial y repitan la prueba.

### Entrega de hoy

Cada grupo entrega:

- configuración elegida del modelo;
- 5 intenciones de su chatbot;
- 5 pruebas con tono detectado e instrucción generada;
- una conversación corta donde se vea el uso del historial.

## 11. Cierre

Lo importante de esta clase:

- No necesitamos corregir todo el texto del usuario.
- La pregunta original se manda a la IA.
- La normalización se usa para análisis interno.
- El PLN ayuda a detectar tono, saludo e intención.
- El tono detectado se transforma en una instrucción concreta para el modelo.
- El análisis de PLN sirve como pista dentro del prompt.
- Reducir tokens e historial ayuda a mejorar la velocidad.

En la próxima clase empezamos con una parte central del proyecto final:

> selección y preparación de fuentes de conocimiento.